In [2]:
   import pandas as pd
   df = pd.read_csv('cardio_train.csv', sep=';')
   df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [3]:
df = df.drop(columns='id')
df['age'] = (df['age'] / 365.25).round(1)
df['bmi'] = df['weight'] / (df['height'] / 100) ** 2

print(df.shape)
print(df.isnull().sum().sum(), 'пропусков')
df.describe().T

(70000, 13)
0 пропусков


,count,mean,std,min,25%,50%,75%,max
age,70000.0,53.302913,6.755126,29.600000,48.400000,53.900000,58.400000,64.900000
gender,70000.0,1.349571,0.476838,1.000000,1.000000,1.000000,2.000000,2.000000
height,70000.0,164.359229,8.210126,55.000000,159.000000,165.000000,170.000000,250.000000
weight,70000.0,74.205690,14.395757,10.000000,65.000000,72.000000,82.000000,200.000000
ap_hi,70000.0,128.817286,154.011419,-150.000000,120.000000,120.000000,140.000000,16020.000000
ap_lo,70000.0,96.630414,188.472530,-70.000000,80.000000,80.000000,90.000000,11000.000000
cholesterol,70000.0,1.366871,0.680250,1.000000,1.000000,1.000000,2.000000,3.000000
gluc,70000.0,1.226457,0.572270,1.000000,1.000000,1.000000,1.000000,3.000000
smoke,70000.0,0.088129,0.283484,0.000000,0.000000,0.000000,0.000000,1.000000
alco,70000.0,0.053771,0.225568,0.000000,0.000000,0.000000,0.000000,1.000000


In [4]:
n_before = len(df)
df = df[
    df['ap_hi'].between(70, 250)
    & df['ap_lo'].between(40, 150)
    & (df['ap_hi'] > df['ap_lo'])
    & df['height'].between(120, 220)
    & df['weight'].between(30, 200)
]
print(f'Удалено строк: {n_before - len(df)} из {n_before}')

Удалено строк: 1390 из 70000


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,68610.0,53.291554,6.757212,29.600000,48.300000,53.900000,58.400000,64.900000
gender,68610.0,1.348637,0.476542,1.000000,1.000000,1.000000,2.000000,2.000000
height,68610.0,164.413249,7.912820,120.000000,159.000000,165.000000,170.000000,207.000000
weight,68610.0,74.114763,14.296269,30.000000,65.000000,72.000000,82.000000,200.000000
ap_hi,68610.0,126.670821,16.678033,70.000000,120.000000,120.000000,140.000000,240.000000
ap_lo,68610.0,81.301866,9.420957,40.000000,80.000000,80.000000,90.000000,150.000000
cholesterol,68610.0,1.364728,0.678930,1.000000,1.000000,1.000000,2.000000,3.000000
gluc,68610.0,1.225783,0.571638,1.000000,1.000000,1.000000,1.000000,3.000000
smoke,68610.0,0.087976,0.283261,0.000000,0.000000,0.000000,0.000000,1.000000
alco,68610.0,0.053360,0.224751,0.000000,0.000000,0.000000,0.000000,1.000000


In [6]:
n_before = len(df)
df = df[df['bmi'].between(15, 60)]
print(f'Удалено строк: {n_before - len(df)}')

Удалено строк: 43


In [7]:
print(df['cardio'].value_counts(normalize=True))

df['age_group'] = pd.cut(df['age'], bins=[29, 40, 50, 55, 60, 65])
for col in ['age_group', 'cholesterol', 'gluc', 'smoke', 'active']:
    print(df.groupby(col)['cardio'].mean().round(3), '\n')

cardio
0    0.505301
1    0.494699
Name: proportion, dtype: float64
age_group
(29, 40]    0.234
(40, 50]    0.376
(50, 55]    0.462
(55, 60]    0.562
(60, 65]    0.671
Name: cardio, dtype: float64 

cholesterol
1    0.435
2    0.596
3    0.762
Name: cardio, dtype: float64 

gluc
1    0.476
2    0.588
3    0.618
Name: cardio, dtype: float64 

smoke
0    0.497
1    0.469
Name: cardio, dtype: float64 

active
0    0.533
1    0.485
Name: cardio, dtype: float64 



In [9]:
from sklearn.metrics import mutual_info_score

cat_cols = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']
num_cols = ['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi']

# Mutual information для категориальных
mi = df[cat_cols].apply(lambda s: mutual_info_score(s, df['cardio']))
print(mi.sort_values(ascending=False).round(4), '\n')

# Корреляция с целевой для числовых
print(df[num_cols + ['cardio']].corr()['cardio'].drop('cardio')
      .sort_values(ascending=False).round(3))

cholesterol    0.0254
gluc           0.0042
active         0.0007
smoke          0.0001
alco           0.0000
gender         0.0000
dtype: float64 

ap_hi     0.428
ap_lo     0.341
age       0.240
bmi       0.192
weight    0.181
height   -0.012
Name: cardio, dtype: float64


In [10]:
from sklearn.model_selection import train_test_split

df = df.drop(columns='age_group')

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train['cardio'].values
y_val = df_val['cardio'].values
y_test = df_test['cardio'].values

for d in (df_train, df_val, df_test):
    del d['cardio']

print(len(df_train), len(df_val), len(df_test))

41139 13714 13714


In [12]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_lr.fit(X_train, y_train)
print('LogReg AUC:', roc_auc_score(y_val, model_lr.predict_proba(X_val)[:, 1]))

LogReg AUC: 0.7883041259208436


In [13]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                            min_samples_leaf=5, n_jobs=-1, random_state=1)
rf.fit(X_train, y_train)
print('RF AUC:', roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1]))

xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                    random_state=1, n_jobs=-1)
xgb.fit(X_train, y_train)
print('XGB AUC:', roc_auc_score(y_val, xgb.predict_proba(X_val)[:, 1]))

RF AUC: 0.7964937364251492
XGB AUC: 0.7970342630766101


In [14]:
results = []

for C in [0.01, 0.1, 1, 10]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000))
    m.fit(X_train, y_train)
    auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
    results.append(('logreg', f'C={C}', round(auc, 4)))

In [15]:
for depth in [6, 8, 10, 14]:
    for leaf in [5, 20]:
        m = RandomForestClassifier(n_estimators=200, max_depth=depth,
                                   min_samples_leaf=leaf, n_jobs=-1, random_state=1)
        m.fit(X_train, y_train)
        auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        results.append(('rf', f'depth={depth}, leaf={leaf}', round(auc, 4)))

In [16]:
for depth in [3, 4, 6]:
    for eta in [0.05, 0.1, 0.3]:
        m = XGBClassifier(n_estimators=200, max_depth=depth, learning_rate=eta,
                          random_state=1, n_jobs=-1)
        m.fit(X_train, y_train)
        auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        results.append(('xgb', f'depth={depth}, eta={eta}', round(auc, 4)))

In [17]:
res = pd.DataFrame(results, columns=['model', 'params', 'auc'])
print(res.sort_values('auc', ascending=False).groupby('model').head(3))

     model             params     auc
15     xgb  depth=4, eta=0.05  0.7978
13     xgb   depth=3, eta=0.1  0.7976
11      rf  depth=14, leaf=20  0.7974
18     xgb  depth=6, eta=0.05  0.7973
9       rf  depth=10, leaf=20  0.7967
8       rf   depth=10, leaf=5  0.7965
0   logreg             C=0.01  0.7883
3   logreg               C=10  0.7883
2   logreg                C=1  0.7883


In [18]:
y_full_train = df_full_train['cardio'].values
df_full_train_x = df_full_train.drop(columns='cardio')

dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(df_full_train_x.to_dict(orient='records'))
X_test = dv.transform(df_test.to_dict(orient='records'))

final = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                      random_state=1, n_jobs=-1)
final.fit(X_full_train, y_full_train)
print('Test AUC:', roc_auc_score(y_test, final.predict_proba(X_test)[:, 1]))

Test AUC: 0.7962765740083443
